In [1]:
import os
import re
import numpy as np
import pandas as pd
import orjson
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
import hnswlib

/Users/yun/develop/dictycite/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from dotenv import load_dotenv
load_dotenv()

True

# one example

In [3]:
def read_jsonl(path: str):
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield orjson.loads(line)

toy_path = "/Users/yun/develop/BioASQ/example/dense_test/pubmed26n0064.jsonl"
docs = list(read_jsonl(toy_path))
len(docs), docs[0].keys()


(30000,
 dict_keys(['pmid', 'docno', 'title', 'abstract', 'mesh_terms', 'keywords', 'is_deleted']))

In [4]:
def parse_mesh_terms(mesh_terms: str) -> list[str]:
    """
    mesh_terms string like:
    'D000445:Aldehyde Oxidoreductases; D000818:Animals; ...'
    Return just names: ['Aldehyde Oxidoreductases', 'Animals', ...]
    """
    if not mesh_terms:
        return []
    parts = [p.strip() for p in mesh_terms.split(";") if p.strip()]
    names = []
    for p in parts:
        # split 'D000445:Name'
        if ":" in p:
            names.append(p.split(":", 1)[1].strip())
        else:
            names.append(p)
    return names

def build_doc_text(d: dict, include_mesh: bool = False) -> str:
    title = (d.get("title") or "").strip()
    abstract = (d.get("abstract") or "").strip()
    text = title
    if abstract:
        text = f"{title}\n\n{abstract}" if title else abstract

    if include_mesh:
        mesh_names = parse_mesh_terms(d.get("mesh_terms") or "")
        if mesh_names:
            # keep it compact
            text = f"{text}\n\nMeSH: " + "; ".join(mesh_names)

    return text.strip()

include_mesh = False  # try True later

rows = []
for d in docs:
    pmid = str((d.get("pmid") or d.get("docno") or "")).strip()
    if not pmid:
        continue
    if d.get("is_deleted") is True:
        continue
    rows.append({
        "pmid": pmid,
        "text": build_doc_text(d, include_mesh=include_mesh),
        "title": (d.get("title") or "").strip(),
    })

df = pd.DataFrame(rows)
df.head()


,pmid,text,title
0,1913804,Membrane organization of the dystrophin-glycop...,Membrane organization of the dystrophin-glycop...
1,1913805,Efficient processing of an antigenic sequence ...,Efficient processing of an antigenic sequence ...
2,1913806,A new mechanism for coactivation of transcript...,A new mechanism for coactivation of transcript...
3,1913807,Disruption of centromere assembly during inter...,Disruption of centromere assembly during inter...
4,1913808,Temporal comparison of recombination and synap...,Temporal comparison of recombination and synap...


In [10]:
with pd.option_context("display.max_rows", None,
                       "display.max_columns", None,
                       "display.max_colwidth", None,
                       "display.width", None):
    display(df.head(2))

,pmid,text,title
0,1913804,"Membrane organization of the dystrophin-glycoprotein complex.\n\nThe stoichiometry, cellular location, glycosylation, and hydrophobic properties of the components in the dystrophin-glycoprotein complex were examined. The 156, 59, 50, 43, and 35 kd dystrophin-associated proteins each possess unique antigenic determinants, enrich quantitatively with dystrophin, and were localized to the skeletal muscle sarcolemma. The 156, 50, 43, and 35 kd dystrophin-associated proteins contained Asn-linked oligosaccharides. The 156 kd dystrophin-associated glycoprotein contained terminally sialylated Ser/Thr-linked oligosaccharides. Dystrophin, the 156 kd, and the 59 kd dystrophin-associated proteins were found to be peripheral membrane proteins, while the 50 kd, 43 kd, and 35 kd dystrophin-associated glycoproteins and the 25 kd dystrophin-associated protein were confirmed as integral membrane proteins. These results demonstrate that dystrophin and its 59 kd associated protein are cytoskeletal elements that are tightly linked to a 156 kd extracellular glycoprotein by way of a complex of transmembrane proteins.",Membrane organization of the dystrophin-glycoprotein complex.
1,1913805,"Efficient processing of an antigenic sequence for presentation by MHC class I molecules depends on its neighboring residues in the protein.\n\nProcessing of endogenously synthesized proteins generates short peptides that are presented by MHC class I molecules to CD8 T lymphocytes. Here it is documented that not only the sequence of the presented peptide but also the residues by which it is flanked in the protein determine the efficiency of processing and presentation. This became evident when a viral sequence of proven antigenicity was inserted at different positions into an unrelated carrier protein. Not different peptides, but different amounts of the antigenic insert itself were retrieved by isolation of naturally processed peptides from cells expressing the different chimeric proteins. Low yield of antigenic peptide from an unfavorable integration site could be overcome by flanking the insert with oligo-alanine to space it from disruptive neighboring sequences. Notably, the degree of protection against lethal virus disease related directly to the amount of naturally processed antigenic peptide.",Efficient processing of an antigenic sequence for presentation by MHC class I molecules depends on its neighboring residues in the protein.


In [ ]:
model_name = "abhinand/MedEmbed-small-v0.1"  # HF model
model = SentenceTransformer(model_name)
# Optional: enforce max sequence length (model default is usually fine)
model.max_seq_length = 512

# Embed doc texts
doc_texts = df["text"].tolist()

In [5]:
emb = model.encode(
    doc_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # cosine similarity
)

emb = emb.astype(np.float32)
emb.shape

Loading weights: 100%|█| 199/199 [
Batches: 100%|█| 469/469 [06:40<00


(30000, 384)

In [7]:
dim = emb.shape[1]
num_elements = emb.shape[0]

# HNSW params (reasonable starting points)
M = 32
ef_construction = 200

index = hnswlib.Index(space="cosine", dim=dim)
index.init_index(max_elements=num_elements, ef_construction=ef_construction, M=M)

# Add vectors (use integer ids 0..N-1)
index.add_items(emb, ids=np.arange(num_elements))

# Query-time ef (higher = better recall, slower)
index.set_ef(100)

print("HNSW built:", num_elements, "vectors,", dim, "dim")


HNSW built: 30000 vectors, 384 dim


In [9]:
def dense_search(query: str, topk: int = 10):
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    labels, distances = index.knn_query(q_emb, k=topk)
    # cosine space distance in hnswlib is (1 - cosine_sim) depending on space; smaller is better
    hits = []
    for idx, dist in zip(labels[0], distances[0]):
        hits.append((int(idx), float(dist)))
    return hits

def pretty_print_hits(query: str, topk: int = 5):
    hits = dense_search(query, topk=topk)
    print("QUERY:", query)
    for rank, (row_id, dist) in enumerate(hits, 1):
        pmid = df.iloc[row_id]["pmid"]
        title = df.iloc[row_id]["title"]
        print(f"{rank:2d}. pmid={pmid}  dist={dist:.4f}  title={title[:120]}")

pretty_print_hits("What is the prognostic role of alterred thyroid profile after cardiosurgery?", topk=5)


QUERY: What is the prognostic role of alterred thyroid profile after cardiosurgery?
 1. pmid=1929631  dist=0.2302  title=Thyroid hormone changes after cardiovascular surgery and clinical implications.
 2. pmid=1934408  dist=0.2497  title=Risk factors for arrhythmia and death after Mustard operation for simple transposition of the great arteries.
 3. pmid=1940515  dist=0.2664  title=[Changes of hypothalamo-pituitary-thyroid function after open heart surgery--especially evaluated by TRH test].
 4. pmid=1936759  dist=0.2669  title=Concentric left ventricular wall thickening in a patient with primary hypothyroidism. Response to gradual thyroxine repl
 5. pmid=1919395  dist=0.2710  title=Regulation by thyroid status of c-myc, c-fos and H-ras mRNAs in the rat myocardium.


top 1 result is in our ground truth and bm25 failed to get it

## check 1 how many are truncated (>512 words)

In [13]:

# SentenceTransformer -> underlying HF tokenizer
# Works for most ST models
hf_tokenizer = model.tokenizer
max_len = getattr(model, "max_seq_length", 512) or 512  # ST sometimes stores it here

def count_tokens(texts, tokenizer, max_len: int, batch_size: int = 512):
    lengths = []
    truncated = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            padding=False,
            truncation=False,   # count full length
            add_special_tokens=True,
            return_attention_mask=False,
            return_token_type_ids=False,
        )
        # lengths include special tokens
        lens = [len(ids) for ids in enc["input_ids"]]
        lengths.extend(lens)
        truncated.extend([l > max_len for l in lens])
    return np.array(lengths, dtype=np.int32), np.array(truncated, dtype=bool)

texts = df["text"].astype(str).tolist()
tok_len, is_trunc = count_tokens(texts, hf_tokenizer, max_len=max_len)

df_stats = df.copy()
df_stats["tok_len"] = tok_len
df_stats["would_truncate"] = is_trunc

print("Model max_seq_length:", max_len)
print("Docs:", len(df_stats))
print("Would truncate:", int(df_stats["would_truncate"].sum()),
      f"({df_stats['would_truncate'].mean()*100:.2f}%)")

# quick distribution
print("\nToken length quantiles:")
print(df_stats["tok_len"].quantile([0, .5, .75, .9, .95, .99, 1.0]).to_string())


Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors


Model max_seq_length: 512
Docs: 30000
Would truncate: 844 (2.81%)

Token length quantiles:
0.00       4.0
0.50     196.0
0.75     313.0
0.90     411.0
0.95     466.0
0.99     597.0
1.00    1042.0


## NaN/shape check

In [15]:
assert emb.dtype == np.float32
assert emb.ndim == 2
assert not np.isnan(emb).any()
print("emb ok:", emb.shape)



emb ok: (30000, 384)
